# Lewisham · `decision_target_date` (web: "Decision target date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `decision_target_date` for every Lewisham application (paged with `search_after`, sorted by `id`,
because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY`.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "decision_target_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "decision_target_date"])
print("Lewisham applications:", len(df))

Lewisham applications: 29021


In [3]:
value = df["decision_target_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,23864,82.23
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,5157,17.77
total,29021,100.00


Values that are not dates:


,id,decision_target_date


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records:

| | Count | Share |
|---|---|---|
| null | 23,864 | 82.23% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 5,157 | 17.77% |

- **Clean format:** every filled value is a real `DD/MM/YYYY` date.
- **Mostly missing:** only 17.8% have a target date (Brent: 89%). Which records have one
  (by year, source system) is not checked yet.

## Step 2 · Which records have a target date: by `bo_system` and year

Fetch `valid_date`, `decision_date` and `bo_system` too. The year is the year of
`valid_date`, or of `decision_date` when there is no valid_date (legacy records), shown
as "(none)" when both are missing.

In [4]:
COLS = ["id", "decision_target_date", "valid_date", "decision_date", "bo_system"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

td = pd.DataFrame(rows).reindex(columns=COLS)
td["has_target"] = td["decision_target_date"].notna()
valid = pd.to_datetime(td["valid_date"], format="%d/%m/%Y", errors="coerce")
decided = pd.to_datetime(td["decision_date"], format="%d/%m/%Y", errors="coerce")
td["year"] = valid.dt.year.fillna(decided.dt.year).astype("Int64").astype(str).replace("<NA>", "(none)")
td["year_from"] = valid.notna().map({True: "valid_date", False: "decision_date"})

print(pd.crosstab(td["bo_system"].fillna("(none)"), td["has_target"], margins=True).to_string())

has_target              False  True    All
bo_system                                 
CSV Data Import         19519  5157  24676
FME Legacy Data Import   4312     0   4312
Portal                     33     0     33
All                     23864  5157  29021


In [5]:
by_year = pd.crosstab(td["year"], td["has_target"])
by_year["share_with_target_%"] = (100 * by_year[True] / by_year.sum(axis=1)).round(1)
by_year.tail(20)

has_target,False,True,share_with_target_%
year,,,
2007,297,0,0.0
2008,276,0,0.0
2009,183,0,0.0
2010,199,0,0.0
2011,205,0,0.0
2012,149,0,0.0
2013,102,0,0.0
2014,168,0,0.0
2015,168,0,0.0


In [6]:
target = pd.to_datetime(td["decision_target_date"], format="%d/%m/%Y", errors="coerce")
print("Earliest target date:", target.min().date(), "| latest:", target.max().date())
print("Records with a target date, by bo_system:", td.loc[td["has_target"], "bo_system"].value_counts().to_dict())

Earliest target date: 2018-04-26 | latest: 2027-01-29
Records with a target date, by bo_system: {'CSV Data Import': 5157}


### Findings · Step 2

| bo_system | with target date | without |
|---|---|---|
| CSV Data Import | 5,157 | 19,519 |
| FME Legacy Data Import | 0 | 4,312 |
| Portal | 0 | 33 |

- **Only the CSV import has target dates:** legacy records never have one (as in Brent),
  and neither do the 33 `Portal` records (a bo_system value not seen before; not looked at).
- **Only from 2019 on:** no record before 2019 has one; target dates run from 2018-04-26 to
  2027-01-29. Brent's start on 2018-01-24.
- **Even from 2019 on, only 17–27% a year have one** (2019: 27.1%, 2020: 16.8%, …, 2026:
  21.1%). Unlike Brent (mostly filled from 2018), most Lewisham applications have no target
  date in any year. Which ones do is not checked yet.

## Step 3 · Which 2019+ records have a target date

Only records from 2019 on (year of `valid_date`, Step 2) from the CSV import, where target
dates exist at all. The share with a target date is shown by `application_type`,
`decision` and `status`, to see whether target dates belong to one kind of record.

In [7]:
COLS3 = ["id", "decision_target_date", "valid_date", "application_type", "decision", "status", "bo_system"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS3, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

recent = pd.DataFrame(rows).reindex(columns=COLS3)
recent["valid_year"] = pd.to_datetime(recent["valid_date"], format="%d/%m/%Y", errors="coerce").dt.year
recent = recent[(recent["valid_year"] >= 2019) & (recent["bo_system"] == "CSV Data Import")].copy()
recent["has_target"] = recent["decision_target_date"].notna()
print("CSV records from 2019 on:", len(recent), "| with target date:", int(recent["has_target"].sum()))


def share_by(column: str) -> pd.DataFrame:
    table = pd.crosstab(recent[column].fillna("(none)"), recent["has_target"])
    table["total"] = table.sum(axis=1)
    table["share_with_target_%"] = (100 * table.get(True, 0) / table["total"]).round(1)
    return table.sort_values("total", ascending=False)


share_by("application_type")

CSV records from 2019 on: 24115 | with target date: 5157


has_target,False,True,total,share_with_target_%
application_type,,,,
All Other,12846,3304,16150,20.5
Householder,4754,1769,6523,27.1
Prior Approval,1358,84,1442,5.8


In [8]:
share_by("decision")

has_target,False,True,total,share_with_target_%
decision,,,,
Approved,12220,4264,16484,25.9
Refused,3841,506,4347,11.6
(none),1064,130,1194,10.9
Withdrawn,758,179,937,19.1
Not Required,675,55,730,7.5
No Objection to Proposal (OBS only),212,11,223,4.9
Closed,98,7,105,6.7
Comment Issued,61,5,66,7.6
Objection Raised to Proposal (OBS only),29,0,29,0.0


In [9]:
share_by("status")

has_target,False,True,total,share_with_target_%
status,,,,
Approved,11355,3684,15039,24.5
Refused,3299,390,3689,10.6
Withdrawn,765,183,948,19.3
Lapsed,467,355,822,43.2
Not Required,661,54,715,7.6
(none),654,1,655,0.2
Completed,395,162,557,29.1
Application Under Consideration,423,129,552,23.4
Dismissed,364,82,446,18.4


### Findings · Step 3

24,115 CSV records from 2019 on; 5,157 (21.4%) have a target date.

- **No kind of record has them all:** every application_type, decision and status has a
  share between 0% and 68%. The target date is not tied to one type or outcome.
- **By type:** Householder 27.1%, All Other 20.5%, Prior Approval 5.8%.
- **By decision:** Approved 25.9%, Withdrawn 19.1%, Refused 11.6%, Not Required 7.5%, opinion
  values 0–7.6%. Approvals have a target date twice as often as refusals.
- **By status:** Commenced 68.4%, Superseded 58.3%, Lapsed 43.2%, Completed 29.1%, pending
  (Under Consideration) 23.4%. Records whose status moved on after approval have one more
  often.
- **So the records with a target date are not a fair sample:** an "on time" figure from
  them would lean towards approvals. Why some records have one is not explained.

## Step 4 · Share with a target date by month of `valid_date`

Same records as Step 3 (CSV import, valid from 2019 on), grouped by the month of
`valid_date`, to see whether target dates come in periods.

In [10]:
recent["valid_month"] = pd.to_datetime(recent["valid_date"], format="%d/%m/%Y").dt.to_period("M")
monthly = recent.groupby("valid_month")["has_target"].agg(records="size", with_target="sum")
monthly["share_%"] = (100 * monthly["with_target"] / monthly["records"]).round(1)
pd.set_option("display.max_rows", 200)
monthly

,records,with_target,share_%
valid_month,,,
2019-01,277,62,22.4
2019-02,318,81,25.5
2019-03,351,91,25.9
2019-04,324,101,31.2
2019-05,299,81,27.1
2019-06,299,84,28.1
2019-07,289,87,30.1
2019-08,245,77,31.4
2019-09,292,79,27.1


### Findings · Step 4

- **No period with full coverage and no gap:** every month from 2019-01 to 2026-08 has
  some records with a target date, and the share stays between 11.6% (2021-01) and 42.9%
  (2026-07). Target dates do not start or stop at a point in time.
- **Steady, not seasonal:** most months are 15–30%; a few peaks (2022-10/11 ~33%,
  2024-06 35.8%, 2026-07 42.9%) with no clear pattern.
- **The last months drop:** 2026-09 9.2%, 2026-10 1.8%, 2026-11 0% (17 records). Recent
  applications may get their target date later; not checked.
- **Time does not explain which records have one.** Together with Step 3: the target date
  is filled for about a fifth of applications, spread over all months and kinds, leaning
  towards approvals.

## Step 5 · Type check and most common values

The Python type of every value in `df` (Step 1), since aggregations hide the JSON type.
Then the most common dates and the count by year of the target date.

In [11]:
types = df["decision_target_date"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Types:", types.value_counts().to_dict())

filled = df["decision_target_date"].dropna()
print()
print("Most common target dates:")
print(filled.value_counts().head(10).to_string())
print()
print("By year of the target date:")
print(pd.to_datetime(filled, format="%d/%m/%Y").dt.year.value_counts().sort_index().to_string())

Types: {'NoneType': 23864, 'str': 5157}

Most common target dates:
decision_target_date
09/10/2026    26
30/11/2026    24
03/03/2023    22
20/12/2024    21
12/09/2024    20
18/09/2026    20
02/09/2024    18
30/04/2021    15
07/10/2022    14
13/01/2023    14

By year of the target date:
decision_target_date
2018      1
2019    703
2020    572
2021    665
2022    795
2023    694
2024    640
2025    501
2026    582
2027      4


### Findings · Step 5

- **Strings or null only** (5,157 strings, 23,864 null); no numbers. Every string is a valid
  `DD/MM/YYYY` date (Step 1).
- **No placeholder date:** the most common target date (09/10/2026) has 26 records; no
  date stands out.
- **By year:** 501–795 a year for 2019–2026; 1 in 2018 (2018-04-26), 4 in 2027 (in the
  future; which applications they are is not checked).

## Step 6 · `decision_target_date` after cleaning

The drops and rules of `semantic/lewisham/cleaning.yaml` applied by hand (as in the status
notebook, Step 5), then the counts `decision_target_date.yaml` gives.

In [12]:
import yaml

rules = yaml.safe_load((PROJECT_ROOT / "semantic/lewisham/cleaning.yaml").read_text())
COLS = ["id", "lpa_app_no", "status", "decision", "decision_date", "valid_date", "decision_target_date",
        "description", "bo_system"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]
raw = pd.DataFrame(rows).reindex(columns=COLS)
for col in COLS[1:]:
    raw[col] = raw[col].map(lambda v: v.strip() or None if isinstance(v, str) else None)

keep = pd.Series(True, index=raw.index)
for rule in rules["all_records"]:
    if rule["action"] == "drop_records":
        hit = raw["id"].isin(rule.get("ids", []))
        if "when_all_null" in rule:
            hit |= raw[rule["when_all_null"]].isna().all(axis=1)
        keep &= ~hit
c = raw[keep].copy()
portal = next(r for r in rules["all_records"] if r["id"] == "outcome_from_portal")
fill = c["id"].isin(portal["outcomes"]) & c["status"].isna() & c["decision"].isna()
for col in ["status", "decision"]:
    c.loc[fill, col] = c.loc[fill, "id"].map(portal["outcomes"])
for rule in rules["fields"]["status"]:
    if rule["action"] == "map":
        c.loc[c["status"] == rule["from"], "status"] = rule["to"]
    elif rule["action"] == "copy_when_null":
        hit = c["status"].isna() & c["decision"].notna()
        c.loc[hit, "status"] = c.loc[hit, "decision"]
    elif rule["action"] == "fill_null":
        c.loc[c["status"].isna(), "status"] = rule["to"]
for rule in rules["fields"]["decision"]:
    c.loc[c["decision"] == rule["from"], "decision"] = rule["to"]
for col in ["decision_date", "valid_date", "decision_target_date"]:
    c[col + "_d"] = pd.to_datetime(c[col], format="%d/%m/%Y", errors="coerce")
print("Records after cleaning:", len(c))

has = c["decision_target_date_d"].notna()
print("With a target date:", int(has.sum()), "| without:", int((~has).sum()), f"({100 * (~has).mean():.1f}% without)")
recent = c[(c["valid_date_d"].dt.year >= 2019) & (c["bo_system"] == "CSV Data Import")]
print("CSV from 2019:", len(recent), f"| with target date: {100 * recent['decision_target_date_d'].notna().mean():.1f}%")
for d in ["Approved", "Refused"]:
    sub = recent[recent["decision"] == d]
    print(f"  {d}: {100 * sub['decision_target_date_d'].notna().mean():.1f}%")
pending = c[c["status"] == "Application Under Consideration"]
print("Pending:", len(pending), "| with a target date:", int(pending["decision_target_date_d"].notna().sum()))
print("Range:", c["decision_target_date_d"].min().date(), "-", c["decision_target_date_d"].max().date())

Records after cleaning: 27765
With a target date: 5139 | without: 22626 (81.5% without)
CSV from 2019: 23387 | with target date: 22.0%
  Approved: 25.9%
  Refused: 11.6%
Pending: 481 | with a target date: 113
Range: 2018-04-26 - 2027-01-29


### Findings · Step 6

27,765 records after cleaning:
- **With a target date: 5,139; without: 22,626 (81.5%).**
- CSV import from 2019: 22.0% have one; Approved 25.9%, Refused 11.6% (unchanged).
- **Pending: 481, of which 113 have a target date** (not 129 of 552).
- Range unchanged: 26/04/2018 to 29/01/2027.